In [ ]:
!nvidia-smi

Sun Nov 22 12:21:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U ultralytics==8.4.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ultralytics]


In [ ]:
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# paths
ZIP_PATH = "/content/drive/MyDrive/MOPG-7.zip"
WORK_DIR = "/content"

# copy & unzip
!cp "$ZIP_PATH" $WORK_DIR/data.zip
!unzip -o $WORK_DIR/data.zip -d $WORK_DIR

Archive:  /content/data.zip
   creating: /content/Dataset/
  inflating: /content/Dataset/classes.txt  
   creating: /content/Dataset/images/
  inflating: /content/Dataset/images/00001.jpg  
  inflating: /content/Dataset/images/000010.jpg  
  inflating: /content/Dataset/images/0000100.jpg  
  inflating: /content/Dataset/images/00001000.jpg  
  inflating: /content/Dataset/images/00001001.jpg  
  inflating: /content/Dataset/images/00001002.jpg  
  inflating: /content/Dataset/images/00001003.jpg  
  inflating: /content/Dataset/images/00001004.jpg  
  inflating: /content/Dataset/images/00001005.jpg  
  inflating: /content/Dataset/images/00001006.jpg  
  inflating: /content/Dataset/images/00001007.jpg  
  inflating: /content/Dataset/images/00001008.jpg  
  inflating: /content/Dataset/images/00001009.jpg  
  inflating: /content/Dataset/images/0000101.jpg  
  inflating: /content/Dataset/images/00001010.jpg  
  inflating: /content/Dataset/images/00001012.jpg  
  inflating: /content/Dataset/imag

In [ ]:
import glob
import os
import random
import shutil
from collections import defaultdict

random.seed(42)

SRC_ROOT = "/content/Dataset"
SPLIT_ROOT = "/content/Data_split"

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

# 1. Identify the dental center from the filename
def identify_center(img_path):
    filename = os.path.splitext(os.path.basename(img_path))[0].lower()

    if filename.startswith("0000"):
        return "Sonia"
    elif filename.startswith("ibn"):
        return "Ibn_Sina"
    elif filename.startswith("nir"):
        return "Niramoy"
    elif filename.startswith("h"):
        return "Health_City"
    else:
        return None


# 2. Find all images
all_images = [
    p for p in glob.glob(
        os.path.join(SRC_ROOT, "**", "*"),
        recursive=True
    )
    if p.lower().endswith(IMG_EXTS)
]

print("Total images found:", len(all_images))


# 3. Group images by dental center
center_groups = defaultdict(list)
unidentified = []

for img in all_images:
    center = identify_center(img)

    if center:
        center_groups[center].append(img)
    else:
        unidentified.append(img)

if unidentified:
    raise ValueError(
        f"{len(unidentified)} images have unidentified center prefixes. "
        f"Examples: {unidentified[:5]}"
    )

expected_centers = {
    "Sonia", "Ibn_Sina", "Niramoy", "Health_City"
}

if set(center_groups.keys()) != expected_centers:
    raise ValueError("One or more expected centers are missing.")

print("\nCenter-wise image counts:")
for center, images in center_groups.items():
    print(center, ":", len(images))


# 4. Find matching YOLO labels
def find_label(img_path):
    stem = os.path.splitext(os.path.basename(img_path))[0]

    candidates = glob.glob(
        os.path.join(SRC_ROOT, "**", stem + ".txt"),
        recursive=True
    )

    labels_only = [
        c for c in candidates
        if os.sep + "labels" + os.sep in c
    ]

    return (labels_only or candidates or [None])[0]


# 5. Split each center independently
splits = {
    "train": [],
    "val": [],
    "test": []
}

for center, images in center_groups.items():

    random.shuffle(images)

    n = len(images)

    n_train = int(n * 0.80)
    n_val = int(n * 0.10)

    train_images = images[:n_train]
    val_images = images[n_train:n_train + n_val]
    test_images = images[n_train + n_val:]

    splits["train"].extend(train_images)
    splits["val"].extend(val_images)
    splits["test"].extend(test_images)

    print(f"\n{center}:")
    print("Train:", len(train_images))
    print("Validation:", len(val_images))
    print("Test:", len(test_images))


# 6. Copy images and YOLO labels
for split_name, images in splits.items():

    img_dir = os.path.join(
        SPLIT_ROOT, "images", split_name
    )

    lbl_dir = os.path.join(
        SPLIT_ROOT, "labels", split_name
    )

    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for img_path in images:

        filename = os.path.basename(img_path)

        shutil.copy2(
            img_path,
            os.path.join(img_dir, filename)
        )

        lbl_path = find_label(img_path)

        if lbl_path:
            shutil.copy2(
                lbl_path,
                os.path.join(lbl_dir, os.path.basename(lbl_path))
            )


# 7. Final summary
print("\n========== FINAL DATA SPLIT ==========")

for split_name, images in splits.items():
    print(f"{split_name}: {len(images)} images")

print("Total:", sum(len(v) for v in splits.values()))

print("\nCenter-wise distribution in each split:")

for split_name, images in splits.items():
    print(f"\n{split_name.upper()}")

    for center in sorted(expected_centers):
        count = sum(
            1 for img in images
            if identify_center(img) == center
        )

        print(f"{center}: {count}")

print("\nCenter-wise splitting completed!")

Total images found: 1941

Center-wise image counts:
Ibn_Sina : 486
Sonia : 1338
Niramoy : 104
Health_City : 13

Ibn_Sina:
Train: 388
Validation: 48
Test: 50

Sonia:
Train: 1070
Validation: 133
Test: 135

Niramoy:
Train: 83
Validation: 10
Test: 11

Health_City:
Train: 10
Validation: 1
Test: 2

========== FINAL DATA SPLIT ==========
train: 1551 images
val: 192 images
test: 198 images
Total: 1941

Center-wise distribution in each split:

TRAIN
Health_City: 10
Ibn_Sina: 388
Niramoy: 83
Sonia: 1070

VAL
Health_City: 1
Ibn_Sina: 48
Niramoy: 10
Sonia: 133

TEST
Health_City: 2
Ibn_Sina: 50
Niramoy: 11
Sonia: 135

Center-wise splitting completed!


In [ ]:
classes_path = "/content/Dataset/classes.txt"
yaml_path = "/content/data.yaml"

with open(classes_path) as f:
    classes = [c.strip() for c in f if c.strip()]

with open(yaml_path, "w") as f:
    f.write(f"path: {SPLIT_ROOT}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("names:\n")
    for i, c in enumerate(classes):
        f.write(f"  {i}: {c}\n")

print("Classes:", classes)
print(open(yaml_path).read())


Classes: ['Missing', 'Dental Crown', 'Root Canal', 'Caries', 'Broken Down', 'Wisdom Teeth', 'No Target Findings']
path: /content/Data_split
train: images/train
val: images/val
names:
  0: Missing
  1: Dental Crown
  2: Root Canal
  3: Caries
  4: Broken Down
  5: Wisdom Teeth
  6: No Target Findings



In [ ]:
!wget -q https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11m.pt
assert os.path.exists("yolo11m.pt"), "YOLOv11 weights missing!"

In [ ]:
# Initialize the YOLO model first
model = YOLO("yolo11m.pt")

results = model.train(
    model="yolo11m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov11m"
)

New https://pypi.org/project/ultralytics/8.4.216 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov11m, nbs=64, nms=False, opset=Non

In [ ]:
metrics_val_v11 = model.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv11m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v11.box.map50)
print("mAP@0.5:0.95:", metrics_val_v11.box.map)
print("Precision   :", metrics_val_v11.box.mp)
print("Recall      :", metrics_val_v11.box.mr)

Ultralytics 8.4.0 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,035,429 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2655.7±1549.3 MB/s, size: 750.0 KB)
val: Scanning /content/Data_split/labels/val.cache... 192 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 192/192 47.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.5it/s 7.8s
                   all        192        975      0.756      0.697        0.7      0.325
               Missing        106        254      0.507      0.354      0.369      0.106
          Dental Crown         62        225      0.916      0.868      0.909      0.474
            Root Canal         63        200      0.695      0.745      0.698      0.204
                Caries         76        114      0.706      0.553      0.546      0.166
           Broken Down         27         

In [ ]:
# Update data.yaml to include the test split path
with open(yaml_path, "r") as f:
    yaml_content = f.read()

if "test:" not in yaml_content:
    # Inject test path after val path
    updated_content = yaml_content.replace("val: images/val\n", "val: images/val\ntest: images/test\n")
    with open(yaml_path, "w") as f:
        f.write(updated_content)
    print("Updated data.yaml with test split path!")

# Run validation on the test split
metrics_test_v11 = model.val(
    data="/content/data.yaml",
    split="test",
    imgsz=640,
    device=0
)

print("=== YOLOv11m TEST RESULTS ===")
print(f"mAP@0.5      : {metrics_test_v11.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_test_v11.box.map:.4f}")
print(f"Precision    : {metrics_test_v11.box.mp:.4f}")
print(f"Recall       : {metrics_test_v11.box.mr:.4f}")

Updated data.yaml with test split path!
Ultralytics 8.4.0 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,035,429 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 137.2±63.9 MB/s, size: 845.2 KB)
val: Scanning /content/Data_split/labels/test... 198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 198/198 408.4it/s 0.5s
val: New cache created: /content/Data_split/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.9it/s 6.8s
                   all        198       1065      0.715      0.686      0.692      0.296
               Missing        109        258      0.589      0.438      0.458      0.134
          Dental Crown         67        247      0.929      0.923      0.948      0.484
            Root Canal         69        267      0.761      0.779      0.753      0.217
                Caries         83    

In [ ]:
import shutil
from google.colab import files
import os

# Define the path for the new zip file
full_content_zip_path = "/content/Split_Data.zip"

# Create a zip archive of the entire /content directory
shutil.make_archive(
    base_name=full_content_zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/content/Data_split",
    base_dir="."
)

print(f"✅ All content zipped at: {full_content_zip_path}")

# Download the created zip file
files.download(full_content_zip_path)

✅ All content zipped at: /content/Split_Data.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model_v10 = YOLO("yolov10m.pt")  # auto-downloads official YOLOv10n weights
model_v10.model.info()


YOLOv10m summary: 288 layers, 16,576,768 parameters, 0 gradients, 64.5 GFLOPs


(288, 16576768, 0, 64.4707968)

In [ ]:
results = model_v10.train(
    model="yolov10m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov10m"
)

New https://pypi.org/project/ultralytics/8.4.216 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov10m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov10m, nbs=64, nms=False, opset=No

In [ ]:
metrics_val_v10 = model_v10.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv10m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v10.box.map50)
print("mAP@0.5:0.95:", metrics_val_v10.box.map)
print("Precision   :", metrics_val_v10.box.mp)
print("Recall      :", metrics_val_v10.box.mr)


Ultralytics 8.4.0 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10m summary (fused): 136 layers, 15,317,221 parameters, 0 gradients, 58.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3245.0±2053.6 MB/s, size: 701.7 KB)
val: Scanning /content/Data_split/labels/val.cache... 192 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 192/192 53.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.9it/s 6.3s
                   all        192        975       0.64      0.714      0.689      0.322
               Missing        106        254      0.452      0.421      0.374      0.122
          Dental Crown         62        225      0.839      0.905      0.891      0.461
            Root Canal         63        200      0.635      0.745      0.688      0.199
                Caries         76        114      0.712      0.606      0.585      0.181
           Broken Down         27        

In [ ]:
metrics_test_v10 = model_v10.val(
    data="/content/data.yaml",
    split="test",
    imgsz=640,
    device=0
)

print("=== YOLOv10m TEST RESULTS ===")
print(f"mAP@0.5      : {metrics_test_v10.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_test_v10.box.map:.4f}")
print(f"Precision    : {metrics_test_v10.box.mp:.4f}")
print(f"Recall       : {metrics_test_v10.box.mr:.4f}")

Ultralytics 8.4.0 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10m summary (fused): 136 layers, 15,317,221 parameters, 0 gradients, 58.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1412.8±201.1 MB/s, size: 901.0 KB)
val: Scanning /content/Data_split/labels/test.cache... 198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 198/198 41.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.5it/s 8.5s
                   all        198       1065      0.668      0.668      0.662      0.307
               Missing        109        258      0.581      0.409      0.462       0.14
          Dental Crown         67        247      0.875      0.931      0.947      0.479
            Root Canal         69        267      0.701      0.678      0.654      0.207
                Caries         83        126        0.7      0.468      0.562      0.218
           Broken Down         31        

In [ ]:
import shutil

zip_path = "/content/runs.zip"

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/content",
    base_dir="runs"
)

print("✅ runs folder zipped at:", zip_path)


✅ runs folder zipped at: /content/runs.zip
